# **인공신경망과 딥러닝**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---



# **1. 신경망의 탄생과 학습 원리**

- 학습목표
    - MP 뉴런과 퍼셉트론의 계산을 수식·코드로 설명할 수 있다
    - 퍼셉트론이 XOR을 풀지 못하는 이유(선형 분리 불가)를 실험으로 확인한다
    - 손실함수·경사하강법·역전파(연쇄법칙)를 손계산 → NumPy → PyTorch autograd 순으로 검증한다



In [ ]:
# [0] 환경 확인 — 30초 안에 끝나는 첫 셀
import numpy as np
import torch
import matplotlib.pyplot as plt
print("numpy", np.__version__, "| torch", torch.__version__)
np.random.seed(42); torch.manual_seed(42)

---
## **1-1. MP 뉴런(1943)**: "뉴런 = 논리 소자"
McCulloch & Pitts(1943)는 뉴런을 **입력의 가중합이 임계값을 넘으면 1을 내보내는 장치**로 모델링했습니다.

$$ y = \begin{cases} 1 & (w_1x_1 + w_2x_2 + b > 0) \\ 0 & (\text{otherwise}) \end{cases} $$

이때 가중치는 **사람이 정했습니다.** (학습 개념 없음)

In [ ]:
# [Lab 1-1] 사람이 가중치를 정한 논리 게이트 (MP 뉴런 방식)
def neuron(x1, x2, w1, w2, b):
    return int(w1 * x1 + w2 * x2 + b > 0)   # 계단(step) 함수

AND  = lambda a, c: neuron(a, c, 0.5, 0.5, -0.7)
OR   = lambda a, c: neuron(a, c, 0.5, 0.5, -0.2)
NAND = lambda a, c: neuron(a, c, -0.5, -0.5, 0.7)

print("x1 x2 | AND OR NAND")
for a, c in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(f" {a}  {c} |  {AND(a,c)}   {OR(a,c)}   {NAND(a,c)}")

---
## **1-2. 퍼셉트론(1958)**: 가중치를 **데이터로부터 학습**
<mark>Rosenblatt(1958)의 핵심은 **틀렸을 때만 가중치를 고치는 학습 규칙**입니다.</mark>

- (단계) **예측**
    - (수식) $\hat y = \mathrm{step}(\mathbf w\cdot\mathbf x + b)$
- (단계) **가중치 수정**
    - $w_i \leftarrow w_i + \eta\,(y-\hat y)\,x_i$
- (단계) **편향 수정**
    - $b \leftarrow b + \eta\,(y-\hat y)$

- 맞히면 $(y-\hat y)=0$ → 변화 없음, 틀리면 정답 쪽으로 경계를 이동
- **퍼셉트론 수렴 정리**: 데이터가 *선형 분리 가능*하면 유한 번 안에 수렴 (Novikoff, 1962)
    - <mark>이 방식은 선형 분리 가능한 데이터만 풀 수 있었다.</mark>

In [ ]:
# [Lab 1-2] 퍼셉트론 학습 규칙 직접 구현
class Perceptron:
    def __init__(self, n_features, lr=0.1, seed=0, w_init=None):
        rng = np.random.default_rng(seed)
        # 가중치: 지정값이 없으면 작은 난수로 시작
        self.w = np.array(w_init, float) if w_init is not None else rng.normal(0, 0.1, n_features)
        self.b = 0.0
        self.lr = lr
        self.history = []                          # 경계선 변화 기록용

    def predict(self, X):
        return (X @ self.w + self.b > 0).astype(int)

    def fit(self, X, y, epochs=20):
        errors = []
        for ep in range(epochs):
            n_err = 0
            for xi, yi in zip(X, y):               # 샘플 하나씩 (온라인 학습)
                err = yi - self.predict(xi[None])[0]
                if err != 0:                       # 틀렸을 때만 수정
                    self.w += self.lr * err * xi
                    self.b += self.lr * err
                    n_err += 1
            errors.append(n_err)
            self.history.append((self.w.copy(), self.b))
            if n_err == 0:                         # 모두 맞히면 종료
                break
        return errors

# 선형 분리 가능한 2차원 데이터 (두 군집)
rng = np.random.default_rng(1)
X0 = rng.normal([-1.5, -1.0], 0.6, (50, 2))
X1 = rng.normal([1.5, 1.0], 0.6, (50, 2))
X = np.vstack([X0, X1])
y = np.array([0] * 50 + [1] * 50)

p = Perceptron(2, lr=0.05, w_init=[-1.0, 0.6])   # 일부러 '틀린 방향'에서 출발
errs = p.fit(X, y, epochs=30)
print("✅ 에폭별 오분류 수:", errs)
print("✅ 최종 가중치:", p.w.round(3), "편향:", round(p.b, 3))

In [ ]:
# 결정 경계가 학습되며 움직이는 모습 시각화 (업데이트 순서대로 색상 변화 적용)
def plot_boundary(ax, w, b, **kw):
    xs = np.linspace(-4, 4, 50)
    if abs(w[1]) > 1e-9:
        ax.plot(xs, -(w[0] * xs + b) / w[1], **kw)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15, zorder=5)

# 색상 맵(Colormap)을 활용하여 에폭이 진행됨에 따라 경계선의 색상이 변하도록 설정
# 빨강(초기) -> 노랑 -> 초록 -> 파랑(최종 직전) 순으로 변화
camp = plt.get_cmap("autumn_r")
n_hist = len(p.history)

for i, (w, b) in enumerate(p.history):
    color = camp(i / max(1, n_hist - 1))
    # 마지막 에폭인 경우 굵기와 강조도를 다르게 설정
    if i == n_hist - 1:
        plot_boundary(axes[0], w, b, color=color, lw=2, ls="--", label=f"epoch {i} (before final)")
    else:
        plot_boundary(axes[0], w, b, color=color, alpha=0.7, lw=1.5, label=f"epoch {i}")

# 학습 완료된 진짜 최종 경계 (검은색 실선)
plot_boundary(axes[0], p.w, p.b, color="black", lw=3, label="final (converged)")

axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)
axes[0].legend(fontsize=9, loc="upper left")
axes[0].set_title("Perceptron boundary update sequence")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

axes[1].plot(errs, marker="o", color="tab:blue", lw=2)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("# errors")
axes[1].set_title("Errors per epoch")
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

### **[실습] 학습률에 따른 에폭수 변화 관찰**
Lab 1-2에서 학습률을 0.01, 1.0으로 바꾸면 수렴 에폭 수가 어떻게 달라지는지 표로 정리하라

| lr | 에폭별 오분류 수| 최종 가중치 |편향 |
|---:|---:|---:|---:|
|0.05| [13, 1, 1, 0] | [0.303 0.902]| -0.05 |
|0.01|  |  |   |
|1.0|  |  |  0.0|

---
## **1-3. XOR 문제(1969)와 첫 번째 AI 겨울**
Minsky & Papert(1969)는 단층 퍼셉트론이 **XOR처럼 선형 분리가 불가능한 문제를 풀 수 없음**을 증명했습니다.
아래에서 직접 확인해 봅시다. 오분류 수가 끝까지 0이 되지 않습니다.

In [ ]:
# [Lab 1-3] 같은 퍼셉트론으로 XOR 학습 시도 → 실패 확인
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

p_xor = Perceptron(2, lr=0.1)
errs_xor = p_xor.fit(X_xor, y_xor, epochs=100)
print("마지막 10에폭 오분류 수:", errs_xor[-10:])
print("예측:", p_xor.predict(X_xor), "| 정답:", y_xor)

<mark>**해결 아이디어: 층을 쌓는다.**</mark>  
XOR = AND(NAND, OR) 처럼 **은닉층**을 두면 풀립니다.
문제는 "은닉층의 가중치를 *어떻게 학습*하느냐"였고, 그 답이 1986년의 **역전파**입니다.

In [ ]:
# 은닉층 2개 뉴런(NAND, OR) + 출력 뉴런(AND) = 손으로 만든 2층 신경망
XOR = lambda a, c: AND(NAND(a, c), OR(a, c))
print([XOR(int(a), int(c)) for a, c in X_xor])   # [0, 1, 1, 0]

In [ ]:
x = torch.randn(32, 784)
l1 = nn.Linear(784, 256)
print(l1.weight.shape, l1.bias.shape)
z1 = l1(x)
print(z1.shape)


# #---------------

---
## **1-4. 경사하강법**: "손실을 줄이는 방향으로 조금씩" 가중치 이동여 값 갱신
퍼셉트론의 계단 함수는 미분이 안 되므로, **미분 가능한 손실**을 정의하고 기울기의 반대 방향으로 이동합니다.

$$ \theta \leftarrow \theta - \eta\,\frac{\partial L}{\partial \theta} $$

학습률 $\eta$가 너무 작으면 느리고, 너무 크면 발산합니다. 직접 확인해 봅시다.

In [ ]:
# [Lab 1-4] 학습률에 따른 경사하강 경로 비교: L(w) = (w - 3)^2
def grad(w):  return 2 * (w - 3)          # 도함수 dL/dw

fig, ax = plt.subplots(figsize=(7, 4))
ws = np.linspace(-1, 7, 200); ax.plot(ws, (ws - 3) ** 2, "k-", lw=1)
for lr, c in [(0.05, "tab:blue"), (0.4, "tab:green"), (1.05, "tab:red")]:
    w, path = -0.5, []
    for _ in range(12):
        path.append(w); w = w - lr * grad(w)
    path = np.array(path)
    ax.plot(path, (path - 3) ** 2, "o-", color=c, ms=4, label=f"lr={lr}")
ax.set_ylim(0, 30); ax.legend(); ax.set_xlabel("w"); ax.set_ylabel("L(w)"); ax.set_title("Gradient descent vs learning rate")
plt.show()

---
## **1-5. 역전파(1986)** = 계산 그래프 위의 연쇄법칙
Rumelhart, Hinton & Williams(1986, *Nature*)가
1986년 데이비드 럼멜하트(David Rumelhart), 제프리 힌튼(Geoffrey Hinton), 로널드 윌리엄스(Ronald Williams)가 _Nature_지에 발표한 논문을 통해 다층 퍼셉트론(MLP)의 모든 층에 대해 편미분(기울기)을 계산하고 이를 **경사하강법으로 한 번에 최적화하는 완전한 학습 체계**를 확립하여 폭발적인 주목을 받게 되었습니다. → 다층 신경망 학습 방법으로 널리 알림



#### 🟠 경사하강법 vs 역전파 알고리즘
| 구분 | 경사하강법 (Gradient Descent) | 역전파 알고리즘 (Backpropagation) |
| :--- | :--- | :--- |
| **최초 제안자 및 연도** | Augustin-Louis Cauchy (1847년) | Rumelhart, Hinton, Williams 등 대중화 (1986년) |
| **개념적 본질** | **파라미터 최적화(Optimization) 알고리즘** | **기울기 연산(Gradient Computation) 알고리즘** |
| **주요 역할** | 계산된 기울기를 활용해 손실을 최소화하는 방향으로 가중치 이동 | 미적분의 연쇄 법칙을 활용해 다층 신경망 각 층의 편미분값 계산 |
| **수학적 기반** | 벡터 미분 및 1차 최적화 이론 | 미적분학의 연쇄 법칙 (Chain Rule) |
| **핵심 수식** | $\mbox{$\theta \leftarrow \theta - \eta \nabla L(\theta)$}$ | $\mbox{$\frac{\partial L}{\partial w_{ij}^{(l)}} = \frac{\partial L}{\partial z_i^{(l)}} \cdot \frac{\partial z_i^{(l)}}{\partial w_{ij}^{(l)}}$}$ |
| **단독 사용 여부** | 볼록 함수(Convex Function) 등 기울기를 직접 구할 수 있는 수식에 단독 적용 가능 | 기울기만 구할 뿐 파라미터를 직접 갱신하지 않으므로 단독으로 학습 불가 |
| **상호 관계** | 역전파가 구해준 기울기($\mbox{$\nabla L$}$)를 넘겨받아 이동 규칙 적용 | 계산한 기울기를 경사하강법(또는 Adam 등 옵티마이저)에 공급 |

- 비유
    - 경사하강법: '기울기를 이용해 값을 갱신하는 엔진',
    - 역전파: '그 기울기를 구해주는 연산기'
- 경사하강법과 역전파 알고리즘으로 <mark>"미분할 수 없는 계단 함수" 대신 부드러운 곡선 형태의 시그모이드(Sigmoid) 함수와 미분 가능한 오차(손실) 함수를 도입하게 되었음</mark>

#### 🟠 **손계산 예제**
$x=2,\ w=0.5,\ b=-0.5,\ y=1$

| 순전파 | 값 | 역전파(국소 미분 × 위에서 온 기울기) |
|---|---|---|
| z = wx+b | $0.5$ | $\partial L/\partial w = \partial L/\partial z \cdot x$ |
| a = $\sigma(z)$ | $0.6225$ | $\partial L/\partial z = \partial L/\partial a \cdot a(1-a)$ |
| L = $(a-y)^2$ | $0.1425$ | $\partial L/\partial a = 2(a-y)$ |

In [ ]:
# [Lab 1-5] 손계산 → 수치미분 → PyTorch autograd 세 가지가 같은지 확인
sig = lambda z: 1 / (1 + np.exp(-z))
x, w, b, y_t = 2.0, 0.5, -0.5, 1.0

# (1) 순전파
z = w * x + b; a = sig(z); L = (a - y_t) ** 2
# (2) 역전파 (연쇄법칙으로 손계산)
dL_da = 2 * (a - y_t)
dL_dz = dL_da * a * (1 - a)
dL_dw, dL_db = dL_dz * x, dL_dz * 1
print(f"z={z:.4f}, a={a:.4f}, L={L:.4f}")
print(f"[손계산]   dL/dw={dL_dw:.6f}, dL/db={dL_db:.6f}")

# (3) 수치 미분: (L(w+h) - L(w-h)) / 2h
h = 1e-5
Lw = lambda w_: (sig(w_ * x + b) - y_t) ** 2
print(f"[수치미분] dL/dw={(Lw(w + h) - Lw(w - h)) / (2 * h):.6f}")

# (4) PyTorch autograd
wt = torch.tensor(w, requires_grad=True)
bt = torch.tensor(b, requires_grad=True)
Lt = (torch.sigmoid(wt * x + bt) - y_t) ** 2
Lt.backward()
print(f"[autograd] dL/dw={wt.grad.item():.6f}, dL/db={bt.grad.item():.6f}")

---
## **1-6. NumPy로 2층 MLP 역전파 직접 구현**
구조: 입력 2 → 은닉 $H$ (tanh) → 출력 1 (sigmoid), 손실은 이진 교차 엔트로피(BCE)

| 순전파 | 역전파 (배치 평균) |
|---|---|
| $Z_1$ = $XW_1$ + $b_1$, $A_1$=$\tanh(Z_1)$ | $dZ_2 = (A_2 - Y)/N$ ← sigmoid+BCE 조합의 깔끔한 결과 |
| $Z_2$ = $A_1W_2$ + $b_2$, $A_2$=$\sigma(Z_2)$ | $dW_2 = A_1^\top dZ_2,\ db_2=\sum dZ_2$ |
| $L$ = $-\frac1N\sum$ [$y\log A_2$ + (1-$y$)$\log$($1-A_2$)] | $dZ_1 = (dZ_2W_2^\top)\odot(1-A_1^2),\ dW_1 = X^\top dZ_1$ |

In [ ]:
# [Lab 1-6] NumPy 2층 MLP
class MLP2:
    def __init__(self, n_in, n_hidden, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden)); self.b1 = np.zeros(n_hidden)
        self.W2 = rng.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, 1)); self.b2 = np.zeros(1)

    def forward(self, X):
        self.X = X
        self.A1 = np.tanh(X @ self.W1 + self.b1)
        self.A2 = 1 / (1 + np.exp(-(self.A1 @ self.W2 + self.b2)))
        return self.A2

    def loss(self, Y):
        A2 = np.clip(self.A2, 1e-9, 1 - 1e-9)
        return -np.mean(Y * np.log(A2) + (1 - Y) * np.log(1 - A2))

    def backward(self, Y):
        N = len(Y)
        dZ2 = (self.A2 - Y) / N
        self.dW2 = self.A1.T @ dZ2;          self.db2 = dZ2.sum(0)
        dZ1 = (dZ2 @ self.W2.T) * (1 - self.A1 ** 2)
        self.dW1 = self.X.T @ dZ1;           self.db1 = dZ1.sum(0)

    def step(self, lr):
        for p in ["W1", "b1", "W2", "b2"]:
            setattr(self, p, getattr(self, p) - lr * getattr(self, "d" + p))

Y_xor = y_xor.reshape(-1, 1).astype(float)
net = MLP2(2, 4, seed=3)
losses = []
for ep in range(3000):
    net.forward(X_xor)
    losses.append(net.loss(Y_xor))
    net.backward(Y_xor)
    net.step(lr=0.5)
print("✅ XOR 예측 확률:", net.forward(X_xor).ravel().round(3), "→", (net.A2.ravel() > 0.5).astype(int))
plt.plot(losses)
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("BCE")
plt.title("NumPy MLP on XOR")
plt.show()

In [ ]:
# [Lab 1-7] Gradient check: 내 역전파가 맞는지 수치미분과 비교 (실무 디버깅 기법)
def numerical_grad(net, name, Y, h=1e-5):
    P = getattr(net, name)
    G = np.zeros_like(P)
    for idx in np.ndindex(P.shape):
        old = P[idx]
        P[idx] = old + h; net.forward(net.X); Lp = net.loss(Y)
        P[idx] = old - h; net.forward(net.X); Lm = net.loss(Y)
        P[idx] = old; G[idx] = (Lp - Lm) / (2 * h)
    return G

net.forward(X_xor)
net.backward(Y_xor)
for name in ["W1", "W2"]:
    ana, num = getattr(net, "d" + name), numerical_grad(net, name, Y_xor)
    rel = np.linalg.norm(ana - num) / (np.linalg.norm(ana) + np.linalg.norm(num) + 1e-12)
    print(f"{name}: 상대오차 = {rel:.2e}  (1e-7 이하면 정상)")

1986년 Rumelhart, Hinton, Williams의 역전파 논문 이후 복잡한 심층 신경망 구현 시 그래디언트 구현 오류가 빈번히 발생하자, 수치 미분(유한 차분법, Finite Difference Method)을 이용한 상대 오차(Relative Error) 검증 공식이 표준 진단 도구로 정립되었습니다. 컴퓨터 연산에서 부동소수점 정밀도(FP32/FP64) 한계로 인해 완전한 0이 나오기 어려우므로 오차 임계치($10^{-7}$ 기준)를 통해 정상 여부를 판별합니다.

In [ ]:
# [Lab 1-8] 같은 가중치로 PyTorch autograd 기울기와 일치하는지 검증
Xt, Yt = torch.tensor(X_xor), torch.tensor(Y_xor)
W1 = torch.tensor(net.W1, requires_grad=True)
b1 = torch.tensor(net.b1, requires_grad=True)
W2 = torch.tensor(net.W2, requires_grad=True)
b2 = torch.tensor(net.b2, requires_grad=True)
out = torch.sigmoid(torch.tanh(Xt @ W1 + b1) @ W2 + b2)
loss = torch.nn.functional.binary_cross_entropy(out, Yt)
loss.backward()
net.forward(X_xor)
net.backward(Y_xor)
print("dW1 일치:", np.allclose(W1.grad.numpy(), net.dW1), "| dW2 일치:", np.allclose(W2.grad.numpy(), net.dW2))
print("→ PyTorch의 loss.backward()가 바로 우리가 손으로 짠 역전파를 자동으로 해 준다!")

### **[실습] 은닉층 활성화 함수 변경에 따른 수정 부분**
`MLP2`의 은닉 활성화를 tanh → sigmoid로 바꾸고 역전파 식의 어느 부분이 바뀌어야 하는지 수정·검증(gradient check)하라

### **[실습] 은닉 뉴런 갯수에 따른 결정경계 시각화하기**
`sklearn.datasets.make_moons(n_samples=400, noise=0.2)` 데이터를 `MLP2`로 학습하고 결정 경계를 등고선으로 그려라. 은닉 뉴런 2/8/32개를 비교하라

---
## **1-7. 보편 근사 정리(1989)**: 은닉층 하나로 어떤 함수든?
Cybenko(1989)는 **은닉 뉴런이 충분히 많으면** 은닉층 1개짜리 신경망으로 연속 함수를 원하는 만큼 근사할 수 있음을 보였습니다.
단, *존재*를 말할 뿐 **학습으로 찾을 수 있는지, 몇 개가 필요한지**는 말하지 않습니다 → 그래서 "깊게" 쌓는 쪽이 실무에서 효율적입니다.

In [ ]:
# [Lab 1-9] 은닉 뉴런 수에 따른 sin 함수 근사 (PyTorch 맛보기)
import torch.nn as nn
xs = torch.linspace(-3.14, 3.14, 200).unsqueeze(1); ys = torch.sin(2 * xs)

plt.figure(figsize=(7, 4)); plt.plot(xs, ys, "k--", label="target sin(2x)")
for H in [2, 8, 64]:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(1, H), nn.Tanh(), nn.Linear(H, 1))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(1500):
        opt.zero_grad(); l = ((model(xs) - ys) ** 2).mean(); l.backward(); opt.step()
    plt.plot(xs, model(xs).detach(), label=f"H={H}, MSE={l.item():.4f}")
plt.legend(); plt.title("Universal approximation in practice"); plt.show()

# #---------------

---
### 참고문헌
- McCulloch, W. S., & Pitts, W. (1943). A logical calculus of the ideas immanent in nervous activity. *Bulletin of Mathematical Biophysics*, 5, 115–133.
- Rosenblatt, F. (1958). The perceptron. *Psychological Review*, 65(6), 386–408.
- Minsky, M., & Papert, S. (1969). *Perceptrons*. MIT Press.
- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323, 533–536.
- Cybenko, G. (1989). Approximation by superpositions of a sigmoidal function. *Mathematics of Control, Signals and Systems*, 2, 303–314.